<a href="https://colab.research.google.com/github/oj-twri/fpl-predictor/blob/main/03_lightgbm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb

# 1. Load the 2023-24 historical data
url = "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/2023-24/gws/merged_gw.csv"
hist = pd.read_csv(url, encoding="latin-1")

# 2. Sort chronologically
hist = hist.sort_values(["element", "GW", "kickoff_time"]).reset_index(drop=True)

# 3. Recreate the baseline predictions so we can compare LightGBM against them
for window in [3, 5]:
    hist[f"pred_last{window}"] = (
        hist.groupby("element")["total_points"]
        .transform(lambda s: s.shift(1).rolling(window, min_periods=1).mean())
    )

print("Loaded:", hist.shape)

Loaded: (29725, 43)


In [2]:
# Stats whose recent history we want to summarise.
# We keep only the columns that exist in this season's file, to avoid a KeyError.
stat_cols = [
    "total_points", "minutes", "goals_scored", "assists", "bonus", "bps",
    "clean_sheets", "goals_conceded", "saves", "ict_index",
    "expected_goals", "expected_assists"
]
stat_cols = [c for c in stat_cols if c in hist.columns]

# Make sure these columns are numbers, not text
hist[stat_cols] = hist[stat_cols].apply(pd.to_numeric, errors="coerce")

# For each stat, average the PREVIOUS 3 and 5 games for each player.
# shift(1) means a game's features only use games before it (no peeking at the answer)
for col in stat_cols:
    for w in [3, 5]:
        hist[f"{col}_last{w}"] = (
            hist.groupby("element")[col]
            .transform(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
        )

# Minutes played in the previous game (a quick signal for "is he a regular starter?")
hist["minutes_prev"] = hist.groupby("element")["minutes"].shift(1)

# Things we know BEFORE kickoff, so they're safe to use for the current game
hist["position_code"] = hist["position"].map({"GK": 0, "GKP": 0, "DEF": 1, "MID": 2, "FWD": 3})
hist["was_home"] = hist["was_home"].astype(int)  # 1 = home, 0 = away
hist["opponent_team"] = hist["opponent_team"].astype("category")  # which team they face

# Final list of columns the model is allowed to see
feature_cols = [
    c for c in hist.columns
    if (c.endswith("_last3") or c.endswith("_last5")) and not c.startswith("pred_")
]
feature_cols += ["minutes_prev", "position_code", "was_home", "value", "opponent_team"]

print(len(feature_cols), "features")
print("Positions that didn't map:", hist["position_code"].isna().sum())

29 features
Positions that didn't map: 0


In [3]:
# Drop each player's first game (no history, so no features)
model_df = hist.dropna(subset=["pred_last5"]).copy()

# Train on the first part of the season, test on the later part.
# The model never sees the test gameweeks while learning.
train = model_df[model_df["GW"] <= 25]
test = model_df[model_df["GW"] > 25].copy()

print("Train rows:", len(train), " Test rows:", len(test))

Train rows: 17855  Test rows: 11005


In [4]:
model = lgb.LGBMRegressor(
    n_estimators=300,       # number of small decision trees to build
    learning_rate=0.03,     # how big a step each tree takes (small = careful)
    num_leaves=15,          # keeps each tree simple, to avoid memorizing noise
    min_child_samples=50,   # each rule must be backed by at least 50 rows
    subsample=0.8,          # each tree sees a random 80% of rows...
    subsample_freq=1,       # ...(this line switches that on)
    colsample_bytree=0.8,   # ...and a random 80% of features
    random_state=42,        # same results every time you run it
    verbose=-1              # quiet output
)

# Learn the link between the features and the points scored
model.fit(train[feature_cols], train["total_points"])

# Predict the points for every test row
test["pred_lgbm"] = model.predict(test[feature_cols])

In [5]:
def report(df, col):
    err = df["total_points"] - df[col]
    return {"MAE": err.abs().mean(), "RMSE": np.sqrt((err ** 2).mean())}

def avg_rank_corr(df, col):
    # Average over gameweeks of how well the predicted order matches the real order
    return (
        df.groupby("GW")[[col, "total_points"]]
        .apply(lambda g: g.corr(method="spearman").iloc[0, 1])
        .mean()
    )

def compare(df):
    cols = ["pred_last5", "pred_lgbm"]
    out = pd.DataFrame({c: report(df, c) for c in cols}).T
    out["Rank corr"] = [avg_rank_corr(df, c) for c in cols]
    return out

print("All players in the test gameweeks:")
display(compare(test))

# Harder, more realistic test: only players who played 60+ minutes last game
starters = test[test["minutes_prev"] >= 60]
print("Starters only:")
display(compare(starters))

All players in the test gameweeks:


,MAE,RMSE,Rank corr
pred_last5,0.908133,1.972717,0.706970
pred_lgbm,0.882286,1.865633,0.694914


Starters only:


,MAE,RMSE,Rank corr
pred_last5,2.403949,3.330169,0.244783
pred_lgbm,2.258143,3.186579,0.275461


In [6]:
# Which features did the model find most useful? (higher = more useful)
importance = pd.Series(
    model.booster_.feature_importance(importance_type="gain"),
    index=feature_cols
).sort_values(ascending=False)

importance.head(10)

,0
minutes_prev,274534.335329
value,50967.858390
opponent_team,48975.189937
total_points_last3,34804.721031
minutes_last3,30074.527086
ict_index_last5,25906.416212
ict_index_last3,24578.839343
bps_last5,16762.357792
minutes_last5,15061.585411
bps_last3,13335.630011
